# Regularized logistic regression

The [regularized regression](regularized-regression.ipynb) chapter penalized
*squared-error* loss for **linear** regression. This chapter penalizes
*cross-entropy* loss for **logistic** regression — same regularization *idea*
(shrink coefficients, discourage overfitting), different loss underneath. It is
**not** a repeat of that chapter.

We extend the hand-rolled logistic gradient descent from the
[logistic regression chapter](logistic-regression.ipynb) with a penalty term.

```{note}
`smartcore`'s `LogisticRegression` has **built-in L2** (`.with_alpha(...)`), but
does not expose L1 / ElasticNet. So we hand-roll all three with one optimizer —
which also keeps the regularization paths directly comparable.
```

In [ ]:
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
:dep plotters-statistical = { version = "0.2.0" }

// 4 features: x0, x1 are real signal; x2, x3 are pure noise (L1 should zero them).
let n = 120usize;
let rows: Vec<Vec<f64>> = (0..n).map(|i| vec![
    ((i * 13) % 20) as f64 * 0.3 - 3.0, ((i * 7) % 20) as f64 * 0.3 - 3.0,
    ((i * 29) % 20) as f64 * 0.3 - 3.0, ((i * 17) % 20) as f64 * 0.3 - 3.0,
]).collect();
let y: Vec<f64> = rows.iter().enumerate().map(|(i, r)| { let noise = (((i * 41) % 11) as f64 - 5.0) * 0.3; if 2.0 * r[0] + 1.5 * r[1] + noise > 0.0 { 1.0 } else { 0.0 } }).collect();

fn sigmoid(z: f64) -> f64 { 1.0 / (1.0 + (-z).exp()) }
// Logistic GD with L2 (ridge) + L1 (lasso, via proximal soft-threshold -> exact zeros).
fn fit(rows: &[Vec<f64>], y: &[f64], l1: f64, l2: f64) -> (f64, Vec<f64>) {
    let (n, p) = (rows.len(), rows[0].len());
    let (mut b, mut w) = (0.0, vec![0.0; p]);
    let (lr, epochs) = (0.3, 2000);
    for _ in 0..epochs {
        let (mut gb, mut gw) = (0.0, vec![0.0; p]);
        for i in 0..n {
            let e = sigmoid(b + (0..p).map(|j| w[j] * rows[i][j]).sum::<f64>()) - y[i];
            gb += e; for j in 0..p { gw[j] += e * rows[i][j]; }
        }
        b -= lr * gb / n as f64;
        for j in 0..p {
            w[j] -= lr * (gw[j] / n as f64 + 2.0 * l2 * w[j]);   // CE + L2 gradient
            if l1 > 0.0 { let t = lr * l1; w[j] = w[j].signum() * (w[j].abs() - t).max(0.0); }  // proximal L1
        }
    }
    (b, w)
}
fn accuracy(rows: &[Vec<f64>], y: &[f64], b: f64, w: &[f64]) -> f64 {
    (0..rows.len()).filter(|&i| ((sigmoid(b + (0..w.len()).map(|j| w[j]*rows[i][j]).sum::<f64>()) > 0.5) as i32 as f64) == y[i]).count() as f64 / rows.len() as f64
}
println!("{} samples, 4 features (x0,x1 signal; x2,x3 noise)", n);

## L2 shrinks; L1 zeros

Fit unregularized, then L2, then L1 (all at a moderate strength) and read the
coefficients. **L2** shrinks *every* coefficient toward zero but keeps them all
non-zero; **L1** drives the least-useful coefficients to **exactly zero** —
producing a *sparse* model, an automatic feature-selection effect L2 doesn't give
you:

In [ ]:
{
    let show = |label: &str, (b, w): (f64, Vec<f64>)| println!("{:<14} b={:+.2}  w=[{:+.2}, {:+.2}, {:+.2}, {:+.2}]", label, b, w[0], w[1], w[2], w[3]);
    show("unregularized", fit(&rows, &y, 0.0, 0.0));
    show("L2 (ridge)",    fit(&rows, &y, 0.0, 0.15));
    show("L1 (lasso)",    fit(&rows, &y, 0.06, 0.0));
    println!("-> L1 produced a SPARSE solution: some coefficients are exactly 0. The path plot shows the order they drop in.");
}

## The regularization path

The single most informative chart for this topic: each coefficient's value as the
L1 strength increases. Every coefficient shrinks and eventually **snaps to exactly
zero**; the order in which they drop reflects how useful each feature is — the
least useful go first. `plotters-statistical`'s `RegularizationPath::new(strengths,
coefficients)` takes the swept strengths and the per-strength coefficient rows,
assigns each feature its own colour and legend entry, and marks where each crosses
zero:

In [ ]:
{
    use plotters::prelude::*;
    use plotters_statistical::RegularizationPath;
    let strengths: Vec<f64> = (0..=24).map(|i| i as f64 * 0.03).collect();
    // coefficients[strength][feature]
    let coefficients: Vec<Vec<f64>> = strengths.iter().map(|&s| fit(&rows, &y, s, 0.0).1).collect();
    evcxr_figure((560, 340), |root| {
        root.fill(&WHITE)?;
        let mut c = ChartBuilder::on(&root)
            .caption("L1 regularization path", ("sans-serif", 14))
            .margin(10).x_label_area_size(30).y_label_area_size(45)
            .build_cartesian_2d(0f64..strengths[strengths.len() - 1], -0.5f64..2.2f64)?;
        c.configure_mesh().x_desc("L1 strength").y_desc("coefficient").draw()?;
        let path = RegularizationPath::new(&strengths, &coefficients)?
            .feature_names(["x0", "x1", "x2", "x3"].iter().copied())
            .stroke_width(2);
        // One draw_series per feature line -> one legend entry each.
        for line in path.lines() {
            let color = line.color();
            let name = line.name().unwrap_or_default().to_string();
            c.draw_series(std::iter::once(line))?
                .label(name)
                .legend(move |(x, y)| PathElement::new(vec![(x, y), (x + 18, y)], color));
        }
        c.configure_series_labels().position(SeriesLabelPosition::UpperRight)
            .border_style(BLACK).background_style(WHITE.mix(0.85)).draw()?;
        Ok(())
    })
}

## Effect on the decision boundary

Regularization's more intuitive payoff for classification is *where it draws the
line*. Using just the two signal features, plot the fitted boundary at weak vs.
strong L2 over the data (points: class 0 red, class 1 blue). Stronger
regularization gives a flatter, less overfit boundary:

In [ ]:
{
    use plotters::prelude::*;
    let two: Vec<Vec<f64>> = rows.iter().map(|r| vec![r[0], r[1]]).collect();
    let (bw, ww) = fit(&two, &y, 0.0, 0.0);      // weak (unregularized)
    let (bs, ws) = fit(&two, &y, 0.0, 0.6);      // strong L2
    // boundary line: b + w0*x0 + w1*x1 = 0  ->  x1 = -(b + w0*x0)/w1
    let line = |b: f64, w: &[f64], x0: f64| -(b + w[0] * x0) / w[1];
    evcxr_figure((460, 340), |root| {
        root.fill(&WHITE)?;
        let mut c = ChartBuilder::on(&root).caption("Decision boundary: weak (green) vs strong L2 (black)", ("sans-serif", 13)).margin(8).x_label_area_size(30).y_label_area_size(40).build_cartesian_2d(-3.2f64..3.2f64, -3.2f64..3.2f64)?;
        c.configure_mesh().x_desc("x0").y_desc("x1").draw()?;
        c.draw_series((0..n).map(|i| Circle::new((two[i][0], two[i][1]), 3, if y[i] > 0.5 { BLUE.filled() } else { RED.filled() })))?;
        c.draw_series(LineSeries::new((-32..=32).map(|k| { let x0 = k as f64 * 0.1; (x0, line(bw, &ww, x0)) }), &GREEN))?;
        c.draw_series(LineSeries::new((-32..=32).map(|k| { let x0 = k as f64 * 0.1; (x0, line(bs, &ws, x0)) }), &BLACK))?;
        Ok(())
    })
}

## Choosing the strength, and a final comparison

Regularization strength is a **hyperparameter** — pick it with
[cross-validation](../01d-evaluation/cross-validation.ipynb) and the
[hyperparameter search](../05b-optimization/hyperparameter-search.ipynb) machinery,
not by eyeballing the path plot. A closing comparison on the same data (accuracy
and how many coefficients survive):

In [ ]:
{
    let row = |label: &str, l1: f64, l2: f64| {
        let (b, w) = fit(&rows, &y, l1, l2);
        let nonzero = w.iter().filter(|c| c.abs() > 1e-6).count();
        println!("{:<16} accuracy={:.3}  non-zero coefs={}/4", label, accuracy(&rows, &y, b, &w), nonzero);
    };
    row("unregularized", 0.0, 0.0);
    row("L2",            0.0, 0.15);
    row("L1",            0.15, 0.0);
    row("ElasticNet",    0.1, 0.1);
    println!("-> L1/ElasticNet keep accuracy while using fewer features (the sparsity payoff).");
}

That completes the regression arc: [linear](linear-regression.ipynb) →
[multi-output](multi-output-regression.ipynb) → [logistic](logistic-regression.ipynb) →
[regularized linear](regularized-regression.ipynb) → regularized logistic. See
the [crate reference](../appendix/crate-reference.md) for what's library-backed
vs. hand-rolled here.